# F2 · Medida previa del apartado 8: ¿se sale el clic fijo?

El clic de M1 se pone en `start` y no se vuelve a mover. **Esta medida decide si el nivel
M2 existe.**

El criterio es el **test directo**: se toma el pixel del centroide del GT en `start` y se
mira si sigue cayendo dentro de la mascara de la misma unidad en cada frame posterior. Eso
es literalmente lo que le pasa al clic de M1, sin intermediarios ni umbrales.

El cociente de la deriva perpendicular contra el semiancho queda como diagnostico
continuo, y se normaliza por el semiancho **del frame t**: la lesion se ensancha con el
tiempo, asi que dividir por el de `start` infla los frames tardios.

Este notebook no contiene logica: llama a `surco.deriva` y muestra.

### Procedencia

**Qué hay que haber ejecutado antes:** `F1_inventario.ipynb`, que deja `inventario_gt.csv` y
`unidades.csv` en `salidas/proceso/`. No hace falta nada más: este notebook mide sobre el GT,
así que **no lee `anotaciones/clics.json`**: el clic que simula es el centroide del GT en
`start`, no el que puso el autor.

**Qué produce**, en `salidas/proceso/`: `deriva.csv` (una fila por par de la ventana),
`deriva_por_unidad.csv` y `fragmentacion_en_start.csv`. Su número es el que decide si el modo
M2 hace falta, y el apartado 5.1 recoge por qué, haciendo falta, no se puede implementar.

**Corre entero en el entorno `surco`**, sin GPU y sin ningún modelo. El mapa completo del
proyecto está en `GUIA.ipynb`.


In [1]:
import pandas as pd

from surco import config, deriva, inventario

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 80)

In [2]:
der = deriva.construir_deriva()
resumen = deriva.resumen_por_unidad(der)
inv = inventario.construir_inventario()
resumen

,serie,etiqueta,start,n_frames,clic_dentro_en_start,pares_fuera,sale_alguna_vez,primer_frame_fuera,semiancho_start_px,perp_max_px,perp_max_norm,frame_del_maximo,paralela_max_px,giro_eje_max_grados,supera_semiancho
0,1,1,3,18,True,7,True,7.0,8.621981,9.426593,1.507114,20,18.402987,10.000017,True
1,1,2,4,17,True,12,True,10.0,5.961857,12.433441,2.412207,21,9.446760,5.461809,True
2,2,1,4,17,True,8,True,14.0,8.664687,18.061473,2.068370,21,10.867240,16.641333,True
3,2,2,4,17,True,13,True,8.0,6.017291,7.835621,1.204429,18,35.132021,21.685035,True
4,3,1,4,17,True,16,True,5.0,8.254210,12.044736,2.056312,7,24.034395,30.705770,True
5,4,1,3,18,True,15,True,7.0,8.482690,23.107299,2.789169,20,20.570891,4.248623,True
6,4,2,3,18,True,17,True,5.0,8.666313,31.405585,4.044247,21,6.026523,4.278489,True
7,5,1,3,18,True,2,True,14.0,9.760392,11.099953,0.881272,18,17.359177,5.614123,False
8,5,2,3,18,True,11,True,5.0,9.737546,7.341080,0.685409,4,22.357937,9.520749,False
9,5,3,4,17,True,10,True,10.0,7.546096,12.203831,1.315439,15,26.946079,4.900677,True


## El numero

Fraccion de pares en los que el clic fijo cae fuera de la lesion, y fraccion de unidades
en las que se sale alguna vez. **Este es el numero que va a la memoria.**

In [3]:
deriva.fracciones_del_clic_fijo(der)

pares_fuera                       183.000000
pares_totales                     295.000000
fraccion_pares_fuera                0.620339
unidades_que_se_salen              15.000000
unidades_totales                   17.000000
fraccion_unidades_que_se_salen      0.882353
dtype: float64

### Unidad a unidad

`clic_dentro_en_start` avisa de un caso degenerado: si el centroide del GT no cae dentro
de su propia mascara ya en `start`, esa unidad no mide deriva, mide que el centroide de
una lesion curvada se queda en el hueco. El clic real del protocolo lo pone una persona
sobre el surco, asi que ahi no habria problema; en el test automatico si.

In [4]:
resumen[
    [
        "serie",
        "etiqueta",
        "start",
        "n_frames",
        "clic_dentro_en_start",
        "pares_fuera",
        "sale_alguna_vez",
        "primer_frame_fuera",
        "perp_max_norm",
    ]
]

,serie,etiqueta,start,n_frames,clic_dentro_en_start,pares_fuera,sale_alguna_vez,primer_frame_fuera,perp_max_norm
0,1,1,3,18,True,7,True,7.0,1.507114
1,1,2,4,17,True,12,True,10.0,2.412207
2,2,1,4,17,True,8,True,14.0,2.068370
3,2,2,4,17,True,13,True,8.0,1.204429
4,3,1,4,17,True,16,True,5.0,2.056312
5,4,1,3,18,True,15,True,7.0,2.789169
6,4,2,3,18,True,17,True,5.0,4.044247
7,5,1,3,18,True,2,True,14.0,0.881272
8,5,2,3,18,True,11,True,5.0,0.685409
9,5,3,4,17,True,10,True,10.0,1.315439


In [5]:
sano = der[der["clic_dentro_en_start"]]
print("dejando fuera las unidades cuyo centroide ya estaba fuera en start:")
print(deriva.fracciones_del_clic_fijo(sano).to_string())

dejando fuera las unidades cuyo centroide ya estaba fuera en start:
pares_fuera                       166.000000
pares_totales                     277.000000
fraccion_pares_fuera                0.599278
unidades_que_se_salen              14.000000
unidades_totales                   16.000000
fraccion_unidades_que_se_salen      0.875000


## El cociente, como diagnostico continuo

No es el criterio, pero dice *cuanto* se sale y no solo *si* se sale. Va normalizado por
el semiancho del frame t. Primero el peor caso de cada unidad, despues los pares uno a
uno, donde se ve que la deriva se acumula con el tiempo y no aparece de golpe.

In [6]:
pd.DataFrame(
    {
        "por unidad (peor caso)": resumen["perp_max_norm"].describe(),
        "por par": der["perpendicular_norm"].describe(),
    }
)

,por unidad (peor caso),por par
count,17.000000,295.000000
mean,1.650662,0.913604
std,0.987647,0.744473
min,0.437938,0.000979
25%,0.881272,0.344438
50%,1.315439,0.680565
75%,2.377459,1.376594
max,4.044247,4.044247


In [7]:
der.groupby("frame")["perpendicular_norm"].mean().rename("cociente medio por frame")

frame
4     0.316590
5     0.401225
6     0.525729
7     0.725537
8     0.775447
9     0.748412
10    0.703510
11    0.787383
12    1.018816
13    0.945899
14    0.990091
15    1.015786
16    0.995067
17    1.072759
18    1.244741
19    1.237249
20    1.334871
21    1.313180
Name: cociente medio por frame, dtype: float64

## En pixeles, como diagnostico

La componente paralela al surco es la inofensiva: el clic se desliza a lo largo de la
lesion pero sigue dentro.

In [8]:
resumen[["semiancho_start_px", "perp_max_px", "paralela_max_px"]].describe()

,semiancho_start_px,perp_max_px,paralela_max_px
count,17.000000,17.000000,17.000000
mean,9.402198,17.106869,18.377030
std,2.623924,13.668911,7.598772
min,5.961857,5.208984,6.026523
25%,8.254210,9.877758,11.315053
50%,8.666313,12.203831,18.402987
75%,9.760392,18.061473,22.357937
max,16.258462,59.519154,35.132021


## La serie 1

Es donde F1 encontro el mayor salto de centroide entre frames consecutivos, 17.8 px. Aqui
se ve en que direccion iba ese salto.

In [9]:
der[der["serie"] == 1]

,serie,etiqueta,start,frame,clic_dentro,clic_dentro_en_start,paralela_px,perpendicular_px,semiancho_start_px,semiancho_frame_px,perpendicular_norm,giro_eje_grados
0,1,1,3,4,True,True,2.235871,-0.309907,8.621981,8.517772,0.036384,3.114929
1,1,1,3,5,True,True,-0.051288,-1.956061,8.621981,11.273642,0.173507,0.285959
2,1,1,3,6,True,True,-1.496630,-4.228441,8.621981,13.084159,0.323173,1.943991
3,1,1,3,7,False,True,-13.324765,-7.892622,8.621981,9.732078,0.810990,1.379942
4,1,1,3,8,True,True,-8.137363,-2.724794,8.621981,7.893257,0.345205,3.294259
5,1,1,3,9,True,True,-11.916120,-5.775305,8.621981,7.575293,0.762387,6.351620
6,1,1,3,10,True,True,-7.674077,-4.830630,8.621981,8.214024,0.588095,4.298737
7,1,1,3,11,True,True,-11.442527,-4.944085,8.621981,7.887826,0.626799,5.139528
8,1,1,3,12,True,True,-15.838492,-6.675809,8.621981,8.312822,0.803074,5.620084
9,1,1,3,13,True,True,-13.259288,-5.082716,8.621981,7.769208,0.654213,6.833851


## Cuanto gira el eje

La base de la descomposicion se fija en `start`. Si el eje de la lesion girase mucho, esa
eleccion dejaria de ser inocua.

In [10]:
resumen.nlargest(5, "giro_eje_max_grados")[
    ["serie", "etiqueta", "giro_eje_max_grados", "perp_max_norm"]
]

,serie,etiqueta,giro_eje_max_grados,perp_max_norm
4,3,1,30.705770,2.056312
11,7,1,29.274550,2.377459
3,2,2,21.685035,1.204429
13,9,1,17.360515,1.154223
2,2,1,16.641333,2.068370


## Concordancia entre el test directo y el cociente

Si el cociente fuera un buen sustituto del test directo, las dos columnas coincidirian.

In [11]:
pd.crosstab(
    resumen["sale_alguna_vez"],
    resumen["supera_semiancho"],
    rownames=["el clic se sale"],
    colnames=["cociente > 1"],
)

cociente > 1,False,True
el clic se sale,,
False,2,0
True,4,11


In [12]:
viejo = deriva.norma_con_semiancho_de_start(der)
print("cuanto inflaba normalizar por el semiancho de start:")
print(f"  media con el semiancho de start : {viejo.mean():.3f}")
print(f"  media con el semiancho del frame: {der['perpendicular_norm'].mean():.3f}")
print(f"  pares en que el de start sale mayor: {(viejo > der['perpendicular_norm']).mean():.1%}")

cuanto inflaba normalizar por el semiancho de start:
  media con el semiancho de start : 1.001
  media con el semiancho del frame: 0.914
  pares en que el de start sale mayor: 62.7%


## El confuso: fragmentacion

El centroide de una mascara partida se mueve aunque la celula este quieta, porque basta
con que desaparezca un fragmento de un extremo. F1 encontro que el 31.7 % de los pares
estan partidos y que la fragmentacion crece con el tiempo, asi que hay que mirar si la
deriva medida es real o es un artefacto de la forma.

In [13]:
cruce = deriva.cruzar_con_inventario(der, inv)
cruce.groupby(config.COLUMNA_FRAGMENTACION)["perpendicular_norm"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
n_comp_c8,,,
1,198,0.759843,0.598494
2,40,1.036050,0.776038
3,35,0.838377,0.654213
4,5,1.211580,1.384895
5,11,2.533974,2.195020
6,3,2.053534,2.006186
7,3,2.728948,2.789169


In [14]:
enteros = cruce[cruce[config.COLUMNA_FRAGMENTACION] == 1]
solo_enteros = deriva.resumen_por_unidad(enteros)
print(f"usando solo los {len(enteros)} pares de una sola componente:")
print(f"  {int(solo_enteros['supera_semiancho'].sum())} de {len(solo_enteros)} unidades superan su semiancho")

usando solo los 198 pares de una sola componente:
  10 de 17 unidades superan su semiancho


## Unidades ya partidas en su frame `start`

`start` es el frame sobre el que F3 anotara los cinco clics. Si la lesion ya esta partida
ahi, los clics **1 y 2** (`un cuarto` y `tres cuartos`, numerados desde 0 como en
`clics.json`) pueden caer en hueco oscuro y entra la clausula de excepcion del apartado 4.2.

In [15]:
frag_start = inventario.fragmentacion_en_start(inv)
frag_start

,serie,etiqueta,start,n_comp_c8,n_comp_c4,area_px2,elongacion,semiancho_px,partida_en_start
0,1,1,3,1,1,1312,9.022163,8.621981,False
1,1,2,4,1,1,854,8.725307,5.961857,False
2,2,1,4,1,1,1222,6.993797,8.664687,False
3,2,2,4,2,2,1057,11.500464,6.017291,True
4,3,1,4,1,1,806,5.527636,8.254210,False
5,4,1,3,1,1,2474,13.846314,8.482690,False
6,4,2,3,1,1,2154,9.898963,8.666313,False
7,5,1,3,1,1,1732,8.333052,9.760392,False
8,5,2,3,1,1,1913,10.041654,9.737546,False
9,5,3,4,1,1,1914,11.699916,7.546096,False


In [16]:
frag_start[frag_start["partida_en_start"]]

,serie,etiqueta,start,n_comp_c8,n_comp_c4,area_px2,elongacion,semiancho_px,partida_en_start
3,2,2,4,2,2,1057,11.500464,6.017291,True
15,10,1,3,2,2,3127,8.527716,12.229086,True


## Salidas

In [17]:
der.to_csv(config.DIR_PROCESO / "deriva.csv", index=False)
resumen.to_csv(config.DIR_PROCESO / "deriva_por_unidad.csv", index=False)
frag_start.to_csv(config.DIR_PROCESO / "fragmentacion_en_start.csv", index=False)
print(f"escritos en {config.DIR_PROCESO}")

escritos en E:\ws_Metricas\salidas\proceso
